<a href="https://colab.research.google.com/github/phamnguyenvietkhuong-lgtm/-/blob/main/Copy_of_Untitled21.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
%pip install -q groq gradio
import sqlite3
import pandas as pd

Q0
1. 寫法3
2. LEFT JOIN之後，沒有匹配物品的類別，items欄位會變成NULL。WHERE i.status='待認領'會把NULL的資料過濾掉，造成效果等同INNER JOIN。

Q1
1. 需要Groq API金鑰、SQLite資料庫，必須完整執行mono_bot呼叫AI，不能單獨測這條商業規則。
2. 整個mono_bot函式都需要修改。
3. 商業規則和資料庫、AI、介面混在一起；不容易單元測試，依賴外部API與資料庫。

A2
1. 資料存取層：跟資料庫互動，執行SQL，不寫商業判斷
2. AI解讀層：解析使用者輸入的文字
3. 商業規則層：實作業務規則，可以單獨做測試
4. 介面層：Gradio網頁介面，處理輸入輸出，不能直接呼叫SQL

規則：上層只能呼叫下一層，禁止跨層呼叫

A3

In [13]:
demo = sqlite3.connect(":memory:")

demo.execute("CREATE TABLE categories(name text);")
demo.execute("CREATE TABLE items(id int, category text, status text);")
demo.executemany("INSERT INTO categories VALUES (?);", [('雨傘',), ('水壺',), ('學生證',), ('耳機',), ('鑰匙',), ('其他',)])
demo.executemany("INSERT INTO items VALUES (?, ?, ?);", [(1, '雨傘', '待認領'), (2, '雨傘', '已認領'), (3, '水壺', '待認領')])

sql_A3 = """
SELECT c.name AS 類別, COUNT(i.id) AS 待認領
FROM categories c
LEFT JOIN items i
ON c.name = i.category AND i.status = '待認領'
GROUP BY c.name
"""
display(pd.read_sql(sql_A3, demo))

,類別,待認領
0,其他,0
1,學生證,0
2,水壺,1
3,耳機,0
4,鑰匙,0
5,雨傘,1


A4


In [14]:
def mock_ai_parse(user_text):
    if "認領" in user_text:
        return {"action":"claim", "item_id":1}
    elif "查詢" in user_text:
        return {"action":"query", "category":"雨傘"}
    else:
        return {"action":"unknown"}

def claim_business_rule(item_status):
    if item_status == "已認領":
        raise Exception("這個物品已經被認領，不可以重複認領")
    return "認領成功"

assert claim_business_rule("待認領") == "認領成功"
print("測試1 ✅ 待認領物品可以認領")

try:
    claim_business_rule("已認領")
    print("測試2 ❌ 失敗：沒有擋住重複認領")
except Exception as err:
    print(f"測試2 ✅ {err}")

測試1 ✅ 待認領物品可以認領
測試2 ✅ 這個物品已經被認領，不可以重複認領
